In [1]:
from pyscf import gto, scf, dft, lib
import numpy as np

## Setups

In [2]:
mol = gto.Mole(atom="C; H 1 0.94; H 1 0.94 2 104.5", spin=2, charge=2, basis="def2-TZVP").build()

In [3]:
mf = scf.UKS(mol, xc="TPSS0").run()

converged SCF energy = -37.717313526496  <S^2> = 2.0026514  2S+1 = 3.0017671


In [4]:
arrays = {
    "mo_coeff": np.asarray(mf.mo_coeff, order="C"),
    "mo_occ": np.asarray(mf.mo_occ, order="C"),
    "rdm1": mf.make_rdm1(),
    "coords": mf.grids.coords,
    "weights": mf.grids.weights,
}

In [5]:
mo_coeff = np.asarray(mf.mo_coeff, order="C")
mo_occ = np.asarray(mf.mo_occ, order="C")
rdm1 = mf.make_rdm1()
coords = mf.grids.coords
weights = mf.grids.weights

In [6]:
grids = mf.grids

In [7]:
ni = dft.numint.NumInt()

In [8]:
np.savez("ch2", **arrays)

In [9]:
ao = ni.eval_ao(mol, grids.coords, deriv=2)
rho_rho = np.array([ni.eval_rho(mol, ao[0], rdm1[s], xctype="lda") for s in (0, 1)])
rho_sigma = np.array([ni.eval_rho(mol, ao[0:4], rdm1[s], xctype="gga") for s in (0, 1)])
rho_tau = np.array([ni.eval_rho(mol, ao[0:4], rdm1[s], xctype="mgga", with_lapl=False) for s in (0, 1)])
rho_lapl = np.array([ni.eval_rho(mol, ao[0:10], rdm1[s], xctype="mgga", with_lapl=True) for s in (0, 1)])

In [10]:
rho_rho.shape, lib.fp(rho_rho)

((2, 33736), np.float64(90.16002674074))

In [11]:
rho_sigma.shape, lib.fp(rho_sigma)

((2, 4, 33736), np.float64(369.8178428546348))

In [12]:
rho_tau.shape, lib.fp(rho_tau)

((2, 5, 33736), np.float64(5587.859016487354))

In [13]:
rho_lapl_ = rho_lapl[:, [0, 1, 2, 3, 5, 4], :]
rho_lapl_.shape, lib.fp(rho_lapl_[:, :5, :]), lib.fp(rho_lapl_[:, 5, :])

((2, 6, 33736), np.float64(5587.859016487354), np.float64(-783527.5368333738))